In [1]:
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split

# 🔹 Bước 1: Đọc dữ liệu gốc
df = pd.read_csv("../Dataset/QK-video_subset_5M.csv")  # Cập nhật đường dẫn nếu cần

# 🔹 Bước 2: Lọc những dòng có tương tác thật
df_clicked = df[df["click"] == 1].copy()

# 🔹 Bước 3: Load encoder và scaler
with open("../Model/Deep FM/video/feature_index.pkl", "rb") as f:
    feature_index = pickle.load(f)

with open("../Model/Deep FM/video/scaler.pkl", "rb") as f:
    scaler = pickle.load(f)

# 🔹 Bước 4: Encode categorical features
for col in ['user_id', 'item_id', 'video_category', 'gender', 'age']:
    mapping = feature_index[col]
    df_clicked[col] = df_clicked[col].astype(str).map(mapping).fillna(0).astype(int)

# 🔹 Bước 5: Scale numeric feature
df_clicked["watching_times_scaled"] = scaler.transform(df_clicked[["watching_times"]])

# 🔹 Bước 6: Chia 80% train, 20% test
_, df_test = train_test_split(df_clicked, test_size=0.2, random_state=42)

# 🔹 Bước 7: Chọn cột cần thiết để đánh giá
df_test_final = df_test[["user_id", "item_id", "gender", "age", "click", "video_category", "watching_times_scaled"]]
df_test_final.reset_index(drop=True, inplace=True)

# 🔹 Bước 8: Lưu ra file
df_test_final.to_csv("../Dataset/QK-video_test.csv", index=False)
print(f"✅ Tạo xong file test: {len(df_test_final)} dòng.")

✅ Tạo xong file test: 390114 dòng.


In [2]:
import pandas as pd

# ✅ Load test set đã được encode và scale sẵn
test_df = pd.read_csv("../Dataset/QK-video_test.csv")

# ✅ Group theo user để biết mỗi user đã click những item nào thật sự
actual_items = test_df.groupby("user_id")["item_id"].apply(list).to_dict()

print(f"✅ Số lượng user cần đánh giá: {len(actual_items)}")

✅ Số lượng user cần đánh giá: 204297


In [3]:
import requests

def get_recommendations(gender, age):
    url = "http://localhost:5001/recommend/video"  # Đảm bảo Flask API đang chạy
    payload = {"gender": gender, "age": age}
    try:
        res = requests.post(url, json=payload)
        if res.status_code == 200:
            recs = res.json()["recommendations"]
            return [item["item_id"] for item in recs]
        else:
            return []
    except:
        return []


In [ ]:
from tqdm import tqdm

recommended_items = {}

for user_id in tqdm(actual_items.keys()):
    # Lấy gender & age từ bất kỳ dòng nào của user đó
    row = test_df[test_df["user_id"] == user_id].iloc[0]
    gender, age = row["gender"], row["age"]
    recs = get_recommendations(gender, age)
    recommended_items[user_id] = recs

  3%|▎         | 5228/204297 [20:25:02<119:38:30,  2.16s/it]      

In [ ]:
import math

def hit_rate_at_k(recommended_items, actual_items, k=5):
    hits = 0
    for user, actual in actual_items.items():
        recs = recommended_items.get(user, [])[:k]
        if any(item in actual for item in recs):
            hits += 1
    return hits / len(actual_items)

def precision_at_k(recommended_items, actual_items, k=5):
    total = 0
    for user, actual in actual_items.items():
        recs = recommended_items.get(user, [])[:k]
        total += len([item for item in recs if item in actual]) / k
    return total / len(actual_items)

def recall_at_k(recommended_items, actual_items, k=5):
    total = 0
    for user, actual in actual_items.items():
        recs = recommended_items.get(user, [])[:k]
        total += len([item for item in recs if item in actual]) / len(actual)
    return total / len(actual_items)

def ndcg_at_k(recommended_items, actual_items, k=5):
    def dcg(recs, actual):
        return sum(1 / math.log2(i + 2) if rec in actual else 0 for i, rec in enumerate(recs[:k]))
    
    total = 0
    for user, actual in actual_items.items():
        ideal_dcg = sum(1 / math.log2(i + 2) for i in range(min(len(actual), k)))
        if ideal_dcg == 0:
            continue
        recs = recommended_items.get(user, [])[:k]
        total += dcg(recs, actual) / ideal_dcg
    return total / len(actual_items)

In [ ]:
K = 5
print(f"🎯 HitRate@{K}:    {hit_rate_at_k(recommended_items, actual_items, K):.4f}")
print(f"🎯 Precision@{K}: {precision_at_k(recommended_items, actual_items, K):.4f}")
print(f"🎯 Recall@{K}:    {recall_at_k(recommended_items, actual_items, K):.4f}")
print(f"🎯 NDCG@{K}:      {ndcg_at_k(recommended_items, actual_items, K):.4f}")